# From Structure Tensor To Vector Field

Author: Saccardo Alessia (s212246@dtu.dk)

In many imaging applications, the goal is not only to analyze intensity values, but to understand the local orientation and organization of structures within the data. Examples include fibrous materials, elongated pores, layered textures, or flow-like patterns. In **qim3d**, this type of information is extracted using the *structure tensor* and visualized through *vector fields*.

This notebook demonstrates the qim3d pipeline from volumetric data to structure tensor computation and finally to vector field visualization, highlighting how local orientation information can be extracted and explored in a consistent and interpretable way.

In [1]:
import qim3d 
import numpy as np

`fibers_150x150x150` is a synthetic 150³ volume with clear fiber-like textures and a dominant orientation, ideal for demonstrating structure-tensor-based orientation estimation and vector-field visualization.

In [2]:
vol = qim3d.examples.fibers_150x150x150
print(vol.shape, vol.dtype, vol.min(), vol.max())

(150, 150, 150) uint8 25 223


In [3]:
qim3d.viz.slicer_orthogonal(vol)

The structure tensor is a local descriptor that summarizes how image intensity varies in a neighborhood around each voxel. It is built from spatial gradients and captures how directional the local signal is. 

At each voxel, the structure tensor yields three eigenvalues and corresponding eigenvectors that describe how intensity varies in different directions. The relative sizes of the eigenvalues indicate whether the local structure is surface-like, fiber-like, or isotropic. By selecting the eigenvector associated with the smallest or largest eigenvalue, different types of dominant orientations can be visualized, depending on the structure of interest.

`val, vec = qim3d.processing.structure_tensor` computes the 3D structure tensor of the input volume and returns its eigenvalues and eigenvectors. When smallest=True, it extracts the eigenvector associated with the smallest eigenvalue at each voxel, which represents the local orientation of elongated or fiber-like structures.

In [4]:
val, vec = qim3d.processing.structure_tensor(vol, smallest = True) 

print("Eigenvalues shape:", val.shape)
print("Eigenvectors shape:", vec.shape)

Computing eigenvalues and eigenvectors of the structure tensor, full = False
Eigenvalues shape: (3, 150, 150, 150)
Eigenvectors shape: (3, 150, 150, 150)


## 2D Vector Field Visualization

`qim3d.viz.vectors` visualizes the orientation vectors derived from the structure tensor by overlaying them on a slice of the input volume. It displays the local orientation as arrows and color-coded directions, allowing interactive exploration of how structure orientation varies across slices of the volume.

In [5]:
qim3d.viz.vectors(vol, vec, axis = 2, interactive = True)

### Understanding Grey Regions: Perpendicular Vectors

Let's explore the same vector field along different axes. Notice that when viewing along certain axes, some regions appear **grey** – this occurs when the orientation vectors are **perpendicular to the viewing plane**, making them difficult to visualize as arrows in 2D. This is an important visual cue that indicates the 3D nature of the orientation field.

In [6]:
# View along axis 0 (YZ plane)
# Grey regions here indicate fibers oriented along the X direction
qim3d.viz.vectors(vol, vec, axis = 0, interactive = True)

In [7]:
# View along axis 1 (XZ plane)
# Grey regions indicate vectors perpendicular to this viewing plane
qim3d.viz.vectors(vol, vec, axis = 1, interactive = True)

## 3D Vector Field Visualization

A vector field is obtained by assigning one of these eigenvectors to every voxel in the volume. The resulting field describes how orientation varies throughout space and provides a compact representation of the global organization of the structures present in the data. Such vector fields can then be visualized using arrows to reveal patterns that are not visible in the raw intensity data alone.

`vector_field_3d` visualizes the orientation field as an interactive 3D cone plot. It samples the eigenvectors on a coarser grid, averages local orientations, and displays the strongest directions as cones whose orientation, size, and color reflect the vector direction and magnitude. This provides an intuitive view of global structural organization in the volume.

Three key parameters control the appearance of the plot:
- **`sampling_step`** defines the grid spacing at which vectors are sampled. Larger values yield fewer, more spread-out cones and faster rendering (e.g. `6` samples every 6th voxel along each axis).
- **`max_cones`** caps the total number of cones displayed; when the sampled grid exceeds this limit, only the locations with the highest eigenvalues (strongest orientation signal) are kept.
- **`cone_size`** is a scaling factor for cone length relative to vector magnitude. Increase it to make orientations more visually prominent.

### Example 1: Dense Sampling

First, let's visualize the vector field with relatively dense sampling to capture fine details of the orientation structure.

In [8]:
fig = qim3d.viz.vector_field_3d(vec, val, sampling_step=6, max_cones=50000, cone_size=2, verbose=False)
fig.show()

### Example 3: Sparse Sampling

Sparse sampling is useful for getting an overview of the dominant orientation patterns without overwhelming detail. This makes it easier to identify global trends and large-scale organization. This is obtained by simultaneously increasing the sampling step and reducing the number of cones plotted.

In [9]:
# Sparse sampling: fewer cones, emphasizes main orientation trends
fig = qim3d.viz.vector_field_3d(vec, val, sampling_step=10, max_cones=3000, cone_size=2, verbose=False)
fig.show()

## Summary

The progression from structure tensor to vector fields demonstrates a powerful workflow for analyzing oriented structures in 3D volumes:

1. **Structure Tensor Computation**: Extracts local orientation information at every voxel
2. **2D Visualization**: Allows slice-by-slice inspection, with grey regions indicating perpendicular vectors
3. **3D Visualization**: Reveals global patterns through interactive cone plots with adjustable density

By varying the sampling parameters, you can balance between detailed local information and clear global trends, making this approach versatile for different analysis needs and volume sizes.